# PM03 Development Notebook

The application code belongs in `src/tree_pipeline.py`. Use this notebook to explore the data, test each function as you implement the function, run the pipeline, and answer the questions at the end.

## Setup

Run the following cell first. In Colab, the cell connects Google Drive; change `PROJECT` if the `pm03_start` folder is not directly under My Drive.

In [4]:
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/pm03_start"
    os.chdir(PROJECT)
elif not os.path.exists("src"):
    os.chdir("..")   # Jupyter Lab starts in the notebooks folder

sys.path.insert(0, "src")
%load_ext autoreload
%autoreload 2

import pandas as pd
import tree_pipeline as tp

print("Project folder:", os.getcwd())

Project folder: /Users/nhowes/Desktop/pm03


## Explore the Data

In [5]:
cases = tp.load_cases("data/support_cases.csv")
print("Cases:", len(cases))
cases.head()

Cases: 10000


,case_id,closed_date,support_channel,support_category,contacts_for_case,days_to_resolve,resolved_first_contact,self_service_attempted,delivery_delay_days,order_value,customer_tenure_months,follow_up_complaint,survey_responded,dissatisfied
0,SC00001,2025-01-01,Email,Product,1,1,Yes,No,1,243.61,68,No,No,
1,SC00002,2025-01-01,Phone,Product,1,2,Yes,No,0,80.99,63,No,No,
2,SC00003,2025-01-01,Chat,Payment,2,2,No,No,-2,108.19,27,No,No,
3,SC00004,2025-01-01,Chat,Payment,1,1,No,No,5,19.86,53,No,Yes,No
4,SC00005,2025-01-01,Chat,Return,2,5,No,No,0,82.33,67,No,No,


In [6]:
# Use this cell to explore the columns, values, and dates.







## Test the Functions

Test each function after implementing the function. The following example shows the pattern: call the function and print the result. Decide for yourself what correct output looks like, and write additional tests in the cells below.

In [7]:
# Example test: print the number of responders and the size of each slice
responders = tp.select_responders(cases)
training, validation, test = tp.split_by_date(responders)
print("Responders:", len(responders))
print("Training:", len(training), " Validation:", len(validation), " Test:", len(test))

Responders: 3051
Training: 2007  Validation: 527  Test: 517


### Test `prepare_features`

In [17]:
# Tests for prepare_features

tp.prepare_features(training)




,contacts_for_case,days_to_resolve,delivery_delay_days,order_value,customer_tenure_months,support_channel_Email,support_channel_Phone,support_category_Delivery,support_category_Payment,support_category_Product,support_category_Return,resolved_first_contact_Yes,self_service_attempted_Yes,follow_up_complaint_Yes
3,1,1,5,19.86,53,0,0,0,1,0,0,0,0,0
9,2,2,-2,43.46,79,0,0,0,0,0,1,0,1,0
11,2,1,3,22.00,40,0,0,0,0,0,1,0,0,0
18,3,2,1,90.60,19,0,1,0,0,0,1,0,1,1
19,3,7,-3,98.25,43,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6632,1,1,4,26.66,76,0,0,0,0,0,0,1,0,0
6636,2,3,0,73.00,65,1,0,0,1,0,0,0,0,0
6638,2,3,5,45.83,19,0,1,1,0,0,0,0,1,0
6640,2,2,2,122.00,19,0,1,0,0,1,0,0,0,0


### Test `train_tree`

In [18]:
# Tests for train_tree
y_train = training["dissatisfied"]
model = tp.train_tree(X_train, y_train, depth=3)

print("Train accuracy:", round(model.score(X_train, y_train), 3))  


Train accuracy: 0.912


### Test `compare_depths and choose_depth`

In [10]:

y_train = training[tp.LABEL]
y_valid = validation[tp.LABEL]

X_train = tp.prepare_features(training)
X_valid = tp.prepare_features(validation, columns=X_train.columns)


depths = [1, 2, 3, 5]
result = tp.compare_depths(X_train, y_train, X_valid, y_valid, depths)

for i, depth in enumerate(depths):
    tree = tp.train_tree(X_train, y_train, depth)
    assert result.loc[i, "depth"] == depth
    assert result.loc[i, "training_accuracy"] == round(tree.score(X_train, y_train), 3)
    assert result.loc[i, "validation_accuracy"] == round(tree.score(X_valid, y_valid), 3)

print("compare_depths values match hand-computed accuracies")
result

best_depth = tp.choose_depth(result)
print(f"Best depth chosen: {best_depth}")



compare_depths values match hand-computed accuracies
Best depth chosen: 3


### Test `evaluate_model`

In [11]:
# Tests for evaluate_model

from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

tree = tp.train_tree(X_train, y_train, 3)
result = tp.evaluate_model(tree, X_valid, y_valid)

predicted = tree.predict(X_valid)
true_neg, false_pos, false_neg, true_pos = confusion_matrix(
    y_valid, predicted, labels=["No", "Yes"]
).ravel()

assert len(result) == 1
assert result.loc[0, "true_positives"] == true_pos
assert result.loc[0, "false_positives"] == false_pos
assert result.loc[0, "false_negatives"] == false_neg
assert result.loc[0, "true_negatives"] == true_neg
assert result.loc[0, "accuracy"] == round(accuracy_score(y_valid, predicted), 3)
assert result.loc[0, "precision"] == round(precision_score(y_valid, predicted, pos_label="Yes"), 3)
assert result.loc[0, "recall"] == round(recall_score(y_valid, predicted, pos_label="Yes"), 3)




### Test `score_cases`

In [12]:
# Tests for score_cases
result = tp.compare_depths(X_train, y_train, X_valid, y_valid, depths)

result

,depth,training_accuracy,validation_accuracy
0,1,0.903,0.894
1,2,0.903,0.894
2,3,0.912,0.903
3,5,0.921,0.896


### Test `evaluate_business_value`

In [28]:
# Tests for evaluate_business_value


### Test `extract_rules`

In [14]:
# Tests for extract_rules
rules = tp.extract_rules(model, X_train.columns)
print(rules)

|--- follow_up_complaint_Yes <= 0.50
|   |--- contacts_for_case <= 2.50
|   |   |--- resolved_first_contact_Yes <= 0.50
|   |   |   |--- weights: [545.00, 62.00] class: No
|   |   |--- resolved_first_contact_Yes >  0.50
|   |   |   |--- weights: [583.00, 11.00] class: No
|   |--- contacts_for_case >  2.50
|   |   |--- self_service_attempted_Yes <= 0.50
|   |   |   |--- weights: [80.00, 33.00] class: No
|   |   |--- self_service_attempted_Yes >  0.50
|   |   |   |--- weights: [11.00, 29.00] class: Yes
|--- follow_up_complaint_Yes >  0.50
|   |--- contacts_for_case <= 1.50
|   |   |--- delivery_delay_days <= 4.50
|   |   |   |--- weights: [33.00, 58.00] class: Yes
|   |   |--- delivery_delay_days >  4.50
|   |   |   |--- weights: [2.00, 35.00] class: Yes
|   |--- contacts_for_case >  1.50
|   |   |--- contacts_for_case <= 2.50
|   |   |   |--- weights: [20.00, 227.00] class: Yes
|   |   |--- contacts_for_case >  2.50
|   |   |   |--- weights: [5.00, 273.00] class: Yes



## Run the Pipeline

In [15]:
!python src/main.py

zsh:1: command not found: python


## Questions

Answer each question in the answer cell directly below the question. Base each answer on the outputs of your pipeline.

**Question 1.** List every column that `prepare_features` removes. For each column, explain why the column must not be a feature.

**Answer:** list: case_id, closed_date, survey_responded, and dissatisfied 

Case_id = no predictive signal but it is a label 

closed date = when the case closed and nothing about the experience which would impact the pattern by adding a time element to it 

survey responded = the model already was filtered at the beginning with the column, and including no value would make it more inconsistent 

dissatisfied = a label and learn the answer of the customers satifisfatction while training

**Question 2.** Using `depth_comparison.csv`, explain how training accuracy and validation accuracy change as depth increases, and why the pipeline chose the depth it chose.

**Answer:**

As depth increases, training accuracy will increase; however, validation accuracy will hit a peak value before dropping again. This is because the tree is learning the training data and not how to make predictions at high depth levels. The pipeline will choose depth 3 since it shows the highest validation accuracy, 0.903, and the lowest depth, showing it has learned to predict for new records, not just answer questions about existing records. depth 4 ties depth 3 in accuracy however 3 will have better performance then 4 so it selected.


**Question 3.** Using `tree_rules.txt`, write a client profile for Customer Support: three or four sentences describing the customers most likely to be dissatisfied, in business language.

Follow-up complaints are the biggest indicator of dissatisfaction. Among those with follow-up complaints, the risk of dissatisfaction only grows the more times the customer is in contact, and the least risky group is those with a single contact and a short delivery delay. Among those without a follow-up complaint, most customers are satisfied. The exception is customers who were in contact 3 or more times and attempted self-service, who are likely to be dissatisfied.

**Question 4.** Identify one rule in `tree_rules.txt` that Customer Support should question before acting on the rule. Explain why.

**Answer:** The follow-up complaint rule is the root of the tree, which makes it the most important rule. However, a follow-up complaint may indicate a customer who is already dissatisfied, not a potentially dissatisfied one, which means the rule may provide little new information. It may be more beneficial to create a tree that focuses on customers who still have the potential to be satisfied if a follow-up call occurs or some action is taken.

**Question 5.** Using `business_value.csv`, explain what the tree means for the supervisors' 100 follow-up calls.

**Answer:**

if a supervisor picked at random about 35 of every 100 calls would be to a dissatisfied customer with this tree 98 of 100 customers would be dissatisfied this is a major improvement.
